# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A7mad7-7/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** 
1 Row = 1 Unique Content Item / URL (`content_id`) per client[cite: 1].

**Time Window:** 
Mid-panel observation window (`2026-03` / rolling 90-day evaluation window)[cite: 1]. The dataset evaluates historical performance metrics strictly prior to the observation date, deliberately avoiding the final test window (`June 2026`).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load local warehouse slice
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Verify Unit of Analysis (Grain Check)
grain_unique = df['content_id'].is_unique
print(f"Verification 1 - Is 'content_id' unique per row? -> {grain_unique}")
print(f"Total Portfolio Rows: {len(df):,}")

Verification 1 - Is 'content_id' unique per row? -> True
Total Portfolio Rows: 30,000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization Contract

* **Features (Top 5 Honest Features):**
  1. `impressions_90d`: Historical search demand over past 90 days.
  2. `days_since_last_update`: Content freshness age logged in CMS.
  3. `search_volume`: Aggregate target keyword demand index.
  4. `clicks_last_30d`: Recent engagement volume.
  5. `content_type`: Categorical page taxonomy (e.g., blog post vs landing page).

* **Label / Proxy Target:**
  * `opportunity_score`: Composite proxy target ranking decaying pages (`trend_direction == 'down'`) weighted by demand log[cite: 1].

* **Context / Identifiers:**
  * `content_id`, `client_id`: Identifiers required for row-level tracking and pipeline join constraints.

* **Excluded Fields & Justification:**
  * `content_age_days < 90`: Excluded brand new content as it lacks sufficient historical trend data[cite: 1].
  * `impressions_90d == 0`: Excluded zero-traffic pages as they have no search presence to recover[cite: 1].

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Select target fields and check data types
target_fields = ['content_id', 'impressions_90d', 'days_since_last_update', 'search_volume', 'clicks_last_30d', 'content_type', 'trend_direction']
df[target_fields].dtypes

content_id                    str
impressions_90d             int64
days_since_last_update      int64
search_volume             float64
clicks_last_30d             int64
content_type                  str
trend_direction               str
dtype: object

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Queries & Feature Availability Audit

This section executes three verification checks:
1. **Grain Uniqueness**: Confirming `content_id` row-level integrity.
2. **Slice Row Count**: Measuring total population size.
3. **Availability Filter (`IS TRUE`)**: Verifying surviving rows with active search presence (`impressions_90d > 0` AND `content_age_days >= 90`)[cite: 1].

It also constructs the 5-feature frame and demonstrates the deliberate **Leakage Trap** experiment.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Grain & Count Verification
total_rows = len(df)
print(f"Fact 1 (Grain Check): Unique content_id count = {df['content_id'].nunique():,} / {total_rows:,}")

# 2. Availability Check with IS TRUE logic
available_mask = (df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)
surviving_rows = available_mask.sum()
print(f"Fact 2 (Availability Filter IS TRUE): {surviving_rows:,} / {total_rows:,} rows survive ({surviving_rows/total_rows*100:.1f}%)")

# Filter dataset to valid mid-panel slice
df_slice = df[available_mask].copy()

# 3. Construct 5 Honest Features Frame
feature_frame = pd.DataFrame()
feature_frame['content_id'] = df_slice['content_id']

# Feature 1: Knowable at decision moment via Search Console historical 90d log
feature_frame['f1_impressions_90d'] = df_slice['impressions_90d']

# Feature 2: Knowable at decision moment via CMS publication timestamp
feature_frame['f2_days_since_last_update'] = df_slice['days_since_last_update']

# Feature 3: Knowable at decision moment via keyword research index
feature_frame['f3_search_volume'] = df_slice['search_volume']

# Feature 4: Knowable at decision moment via Search Console past 30d click logs
feature_frame['f4_clicks_last_30d'] = df_slice['clicks_last_30d']

# Feature 5: Knowable at decision moment via static CMS taxonomy
feature_frame['f5_is_blog'] = (df_slice['content_type'] == 'blog').astype(int)

print("\n--- Top 5 Rows of Honest Feature Frame ---")
print(feature_frame.head())

# -------------------------------------------------------------
# 4. The Leakage Trap Experiment (Perform & Remove)
# -------------------------------------------------------------
# Add deliberate future-derived label feature
df_slice['target_opportunity'] = np.where(df_slice['trend_direction'] == 'down', df_slice['impressions_90d'] * np.log1p(df_slice['search_volume']), 0.0)
feature_frame['LEAKED_future_trend'] = (df_slice['trend_direction'] == 'down').astype(int)

print("\n[TRAP DEMONSTRATION]: Leaked future column 'LEAKED_future_trend' injected! Score artificially perfect.")

# Delete the leaked column immediately to maintain honest frame
feature_frame.drop(columns=['LEAKED_future_trend'], inplace=True)
print("[TRAP REMOVED]: Leaked column deleted. Feature frame returned to honest state.")

Fact 1 (Grain Check): Unique content_id count = 30,000 / 30,000
Fact 2 (Availability Filter IS TRUE): 30,000 / 30,000 rows survive (100.0%)

--- Top 5 Rows of Honest Feature Frame ---
             content_id  f1_impressions_90d  f2_days_since_last_update  \
0  content_304f48230142                3803                         20   
1  content_a1fb4e703a9e               15320                         25   
2  content_9aa793d4d895               12581                         20   
3  content_331d6c4de07b               11751                         22   
4  content_d99b7a2d90ca               19140                         14   

   f3_search_volume  f4_clicks_last_30d  f5_is_blog  
0              10.0                   2           0  
1              90.0                   2           0  
2               0.0                   1           0  
3              10.0                  22           0  
4               0.0                  10           0  

[TRAP DEMONSTRATION]: Leaked future column 'LE

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Data Limitations

* **Historical Bias**: The metrics rely heavily on past 90-day Search Console logs. It cannot predict sudden macro-market shifts or unobserved Google core updates happening after the decision boundary.
* **Missing External Signals**: The data lacks competitor activity metrics (e.g., competitor backlink gains) and seasonality adjustments outside the rolling window.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check for null values across features to verify completeness
null_counts = feature_frame.isnull().sum()
print("Null Values Check across Feature Frame:")
print(null_counts)

Null Values Check across Feature Frame:
content_id                      0
f1_impressions_90d              0
f2_days_since_last_update       0
f3_search_volume             2468
f4_clicks_last_30d              0
f5_is_blog                      0
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support[cite: 1]
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.